# Day 2 PM: Document-feature matrices

# LIWCing at SOTUs…

![The Count from Sesame Street holding up a hand and counting.](img/p65-i0.png)

![LIWC2015 marketing page: "DISCOVER LIWC2015" with Buy Academic/Commercial buttons.](img/p66-i0.png)

<div class="side-by-side">

![LIWC-22 marketing page: "INTRODUCING LIWC-22".](img/p67-i0.png)
![Photo of James Pennebaker.](img/p67-i1.png)

</div>

<div class="side-by-side">

![LIWC-22 marketing page.](img/p68-i0.png)
![Photo of James Pennebaker.](img/p68-i1.png)
![Photo of Yla Tausczik.](img/p68-i3.png)
![Johnny Cash performing live, flipping the bird at the camera.](img/p68-i2.png)

</div>

In [ ]:
import re
import pandas
import matplotlib
import matplotlib.pyplot as plt
import sotu

matplotlib.style.use("ggplot")
matplotlib.rcParams["axes.facecolor"] = "#EBEBEB"
matplotlib.rcParams["axes.edgecolor"] = "#EBEBEB"
matplotlib.rcParams["axes.labelcolor"] = "#3C3C3C"
matplotlib.rcParams["xtick.color"] = "#3C3C3C"
matplotlib.rcParams["ytick.color"] = "#3C3C3C"
matplotlib.rcParams["grid.color"] = "white"
matplotlib.rcParams["axes.grid"] = True

sotu_df = sotu.load().reset_index(drop=True)

In [ ]:
# LIWC .dic parser. Tries exact match first, then longest matching prefix wildcard.
# Uses a trie so a single token check is O(token_length), not O(num_prefixes).

def load_liwc(path):
    text = open(path, encoding="utf-8", errors="replace").read()
    blocks = text.split("%", 2)
    categories = {}
    for line in blocks[1].strip().splitlines():
        parts = line.split("\t")
        if len(parts) >= 2 and parts[0].strip().isdigit():
            categories[int(parts[0])] = parts[1].strip()
    exact = {}
    trie = {}
    for line in blocks[2].strip().splitlines():
        parts = line.split("\t")
        if len(parts) < 2:
            continue
        word = parts[0].strip().lower()
        if " " in word or word.startswith("("):
            continue
        cats = []
        for c in parts[1:]:
            c = c.strip()
            if c.isdigit() and int(c) in categories:
                cats.append(int(c))
        if not cats:
            continue
        if word.endswith("*"):
            node = trie
            for ch in word[:-1]:
                node = node.setdefault(ch, {})
            node["$"] = cats
        else:
            exact[word] = cats
    return categories, exact, trie

def liwc_match(token, exact, trie):
    if token in exact:
        return exact[token]
    node = trie
    best = None
    for ch in token:
        if "$" in node:
            best = node["$"]
        if ch not in node:
            return best or []
        node = node[ch]
    if "$" in node:
        best = node["$"]
    return best or []

categories, liwc_exact, liwc_trie = load_liwc("../day-1/dictionaries/liwcdict.dic")
print(f"LIWC dictionary: {len(categories)} categories, {len(liwc_exact)} exact words")

In [ ]:
# Score every SoU. WC=word count, WPS=words per sentence, Sixltr=% words >=6 letters,
# Dic=% tokens classified, each LIWC category as % of total tokens.

def score_document(text):
    sentences = re.split(r"(?<=[.!?])\s+", text)
    sentences = [s for s in sentences if s.strip()]
    tokens = re.findall(r"[A-Za-z']+", text.lower())
    wc = len(tokens)
    if wc == 0:
        return None
    counts = {cat: 0 for cat in categories.values()}
    classified = 0
    sixltr = 0
    for tok in tokens:
        if len(tok) >= 6:
            sixltr += 1
        cats = liwc_match(tok, liwc_exact, liwc_trie)
        if cats:
            classified += 1
            for c in cats:
                counts[categories[c]] += 1
    row = {
        "WC": wc,
        "WPS": wc / max(len(sentences), 1),
        "Sixltr": 100 * sixltr / wc,
        "Dic": 100 * classified / wc,
    }
    for cat in categories.values():
        row[cat] = 100 * counts[cat] / wc
    return row

rows = []
for i in range(len(sotu_df)):
    speech = sotu_df.iloc[i]
    scored = score_document(speech["text"])
    if scored is None:
        continue
    scored["president"] = speech["president"]
    scored["year"] = speech["year"]
    rows.append(scored)

liwc_df = pandas.DataFrame(rows)
print(f"LIWC-scored {len(liwc_df)} SoU addresses, {liwc_df['year'].min()}–{liwc_df['year'].max()}")

## LIWC categories applied to every State of the Union

In [ ]:
preview_cols = ["president", "year", "WPS", "WC",
                "Function", "Pronoun", "I", "We", "Affect"]
print(liwc_df[preview_cols].head(14).to_string(index=False, float_format="%.2f"))

In [ ]:
def plot_liwc_timeseries(category):
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(liwc_df["year"], liwc_df[category],
            marker="o", color="black", linewidth=0.8, markersize=4)
    ax.set_xlabel("year")
    ax.set_ylabel(category)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_timeseries(category, df):
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(df["year"], df[category],
            marker="o", color="black", linewidth=0.8, markersize=4)
    ax.set_xlabel("year")
    ax.set_ylabel(category)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

def plot_liwc_timeseries(category):
    plot_timeseries(category, liwc_df)

In [ ]:
plot_liwc_timeseries("WC")

<div class="image-grid">

![LIWC base categories: Function, Pronoun, Apron, I, We, You, …](img/p70-i0.png)
![MAC-D virtue categories: Family, Group, Reciprocity, Heroism, Deference, Fairness, Property.](img/p70-i1.png)
![nuke.dic categories: nuclear, weapons, proliferation, …](img/p70-i2.png)
![nietzsche.dic with English and German entries (admiration, affect, anger, schnöde, niederträcht*, …).](img/p70-i3.png)

</div>

<!-- demo slide -->

In [ ]:
plot_liwc_timeseries("WPS")

<!-- demo slide -->

In [ ]:
plot_liwc_timeseries("Posemo")

<!-- demo slide -->

In [ ]:
plot_liwc_timeseries("Anx")

<!-- demo slide -->

In [ ]:
plot_liwc_timeseries("Anger")

<!-- demo slide -->

In [ ]:
plot_liwc_timeseries("Sad")

# beyond base LIWC: custom dictionaries

## What about <span class="color-red">moral</span> language?

Custom dictionaries!

**MAC-D** (or **MFD** if you prefer a dictionary based on shitty theory)

<div class="side-by-side">

![Heliyon paper: "Moral universals: a machine-reading analysis of 256 societies" (Alfano, Cheong & Curry, 2024).](img/p80-i1.png)
![Behavior Research Methods paper: "The extended Moral Foundations Dictionary (eMFD)" (Hopp et al., 2021).](img/p80-i2.png)

</div>

👍 **MAC-D** &nbsp;&nbsp;&nbsp; 💩 **MFD**

In [ ]:
macd_categories, macd_exact, macd_trie = load_liwc("../day-1/dictionaries/macdvirtue.dic")
print(f"MAC-D dictionary: {len(macd_categories)} categories, {len(macd_exact)} exact words")
print("Categories:", ", ".join(macd_categories[i] for i in sorted(macd_categories)))

In [ ]:
def score_macd(text):
    tokens = re.findall(r"[A-Za-z']+", text.lower())
    wc = len(tokens)
    if wc == 0:
        return None
    counts = {cat: 0 for cat in macd_categories.values()}
    for tok in tokens:
        cats = liwc_match(tok, macd_exact, macd_trie)
        for c in cats:
            counts[macd_categories[c]] += 1
    row = {cat: 100 * counts[cat] / wc for cat in macd_categories.values()}
    return row

macd_rows = []
for i in range(len(sotu_df)):
    speech = sotu_df.iloc[i]
    scored = score_macd(speech["text"])
    if scored is None:
        continue
    scored["president"] = speech["president"]
    scored["year"] = speech["year"]
    macd_rows.append(scored)

macd_df = pandas.DataFrame(macd_rows)
print(f"MAC-D-scored {len(macd_df)} SoUs")

<!-- demo slide -->

In [ ]:
plot_timeseries("Family", macd_df)

<!-- demo slide -->

In [ ]:
plot_timeseries("Group", macd_df)

<!-- demo slide -->

In [ ]:
plot_timeseries("Reciprocity", macd_df)

<!-- demo slide -->

In [ ]:
plot_timeseries("Heroism", macd_df)

<!-- demo slide -->

In [ ]:
plot_timeseries("Deference", macd_df)

<!-- demo slide -->

In [ ]:
plot_timeseries("Fairness", macd_df)

<!-- demo slide -->

In [ ]:
plot_timeseries("Property", macd_df)

# beyond base LIWC: semantic networks

<div class="side-by-side">

![Photo of John Rupert Firth, linguist.](img/p89-i0.png)

</div>

## You shall know a word by the company that it keeps.

~John Rupert Firth

So you need **context**, e.g., **collocations** or **embeddings**…

<div class="side-by-side">

![AI-painting of three philosophers gathered around a desk debating.](img/p90-i0.png)
![KEEP CALM and RAMSIFY (with a crown emoji).](img/p90-i2.png)

</div>

!["Always has been" astronaut meme: "it's all just matrix multiplication?" / "always has been".](img/p91-i0.png)

![Transpose: a 3×3 coloured matrix mirrored along its diagonal.](img/p92-i0.png)

![Top Gun cockpit view with the jet flying upside-down.](img/p93-i0.png)

## Matrix multiplication is <span class="color-red">not commutative</span>!

$$M \times T \neq T \times M$$

*What question do you want to ask?*

# Live demo!

*(in which Mark takes his life in his hands)*

In [ ]:
import networkx
from itertools import combinations

virtues = ["Family", "Group", "Reciprocity", "Heroism",
           "Deference", "Fairness", "Property"]

def virtue_network(df, ax, title):
    G = networkx.Graph()
    for v in virtues:
        G.add_node(v, prevalence=df[v].mean())
    corr = df[virtues].corr()
    for a, b in combinations(virtues, 2):
        w = max(corr.loc[a, b], 0)
        G.add_edge(a, b, weight=w)
    pos = networkx.circular_layout(G)
    sizes = [G.nodes[n]["prevalence"] * 6000 + 600 for n in G.nodes]
    widths = [G[u][v]["weight"] * 6 for u, v in G.edges]
    networkx.draw_networkx_nodes(G, pos, node_color="#FFC0CB",
                                 node_size=sizes, ax=ax,
                                 edgecolors="#A06070", linewidths=0.5)
    networkx.draw_networkx_labels(G, pos, font_size=10, ax=ax)
    networkx.draw_networkx_edges(G, pos, width=widths,
                                 edge_color="#888888", ax=ax)
    ax.set_title(title)
    ax.axis("off")

<!-- demo slide -->

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)
virtue_network(macd_df[macd_df["year"] < 1900], axes[0], "pre-1900")
virtue_network(macd_df[macd_df["year"] >= 1900], axes[1], "1900–present")
plt.show()

## Before we end, sticky note feedback

- On the green sticky note, write one thing that went well this session.
- On the red sticky note, write one thing that we can improve on, tomorrow.

# Thanks for your attention!